In [1]:
import gc
import io
import os
import json
import time
import uuid
import base64
import logging
import warnings
import traceback
from collections import deque
from datetime import datetime

import cv2
import numpy as np
from PIL import Image
from flask import Flask, request, jsonify, render_template
from flask_cors import CORS

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Model
from tensorflow.keras import layers as KL
from scipy.special import softmax

I0000 00:00:1784127064.128390   12971 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784127064.134804   12971 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1784127066.573398   12971 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784127066.573687   12971 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("nutribone")

In [3]:
CATEGORIES    = ["Normal", "Osteopenia", "Osteoporosis"]
NUM_CLASSES   = len(CATEGORIES)
IMG_SIZE      = (384, 384)
TTA_STEPS     = 4
CONFIG_PATH   = "nutribone_config.json"
MAX_FILE_MB   = 20
MIN_DIM       = 64
MAX_HISTORY   = 20

prediction_history: deque = deque(maxlen=MAX_HISTORY)

In [4]:
@tf.keras.utils.register_keras_serializable(package="NutriBone")
class EfficientNetV2Preprocess(KL.Layer):
    """Rescales [0, 255] → [-1, 1] as expected by EfficientNetV2."""
    def call(self, x):
        return tf.keras.applications.efficientnet_v2.preprocess_input(x)

    def get_config(self):
        return super().get_config()


@tf.keras.utils.register_keras_serializable(package="NutriBone")
class DenseNetPreprocess(KL.Layer):
    """Subtracts ImageNet BGR channel means; used by DenseNet201."""
    def call(self, x):
        return tf.keras.applications.densenet.preprocess_input(x)

    def get_config(self):
        return super().get_config()


@tf.keras.utils.register_keras_serializable(package="NutriBone")
class ConvNeXtPreprocess(KL.Layer):
    """Normalises with ImageNet mean/std as expected by ConvNeXtTiny."""
    def call(self, x):
        return tf.keras.applications.convnext.preprocess_input(x)

    def get_config(self):
        return super().get_config()

In [5]:
@tf.keras.utils.register_keras_serializable(package="NutriBone")
class FocalLoss(keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=0.25, label_smoothing=0.05,
                 reduction="sum_over_batch_size", name="focal_loss", **kwargs):
        super().__init__(reduction=reduction, name=name, **kwargs)
        self.gamma           = gamma
        self.alpha           = alpha
        self.label_smoothing = label_smoothing

    def call(self, y_true, y_pred):
        y_true = y_true * (1 - self.label_smoothing) + self.label_smoothing / NUM_CLASSES
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        ce     = -y_true * tf.math.log(y_pred)
        w      = self.alpha * y_true * tf.pow(1 - y_pred, self.gamma)
        return tf.reduce_mean(tf.reduce_sum(w * ce, axis=1))

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"gamma": self.gamma, "alpha": self.alpha,
                    "label_smoothing": self.label_smoothing})
        return cfg


@tf.keras.utils.register_keras_serializable(package="NutriBone")
class WarmupCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, peak_lr, total_steps, warmup_steps, min_lr=1e-9, **kwargs):
        super().__init__(**kwargs)
        self.peak_lr = float(peak_lr); self.total_steps = float(total_steps)
        self.warmup_steps = float(warmup_steps); self.min_lr = float(min_lr)
    def __call__(self, step):
        import math as _m
        step = tf.cast(step, tf.float32)
        warmup_lr = self.peak_lr * (step / self.warmup_steps)
        p = (step - self.warmup_steps) / (self.total_steps - self.warmup_steps)
        cosine_lr = self.min_lr + 0.5*(self.peak_lr-self.min_lr)*(1+tf.cos(_m.pi*p))
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)
    def get_config(self):
        return {'peak_lr': self.peak_lr, 'total_steps': self.total_steps,
                'warmup_steps': self.warmup_steps, 'min_lr': self.min_lr}

@tf.keras.utils.register_keras_serializable(package="NutriBone")
class EfficientNetV2Preprocess(KL.Layer):
    def call(self, x): return tf.keras.applications.efficientnet_v2.preprocess_input(x)
    def get_config(self): return super().get_config()

@tf.keras.utils.register_keras_serializable(package="NutriBone")
class DenseNetPreprocess(KL.Layer):
    def call(self, x): return tf.keras.applications.densenet.preprocess_input(x)
    def get_config(self): return super().get_config()

@tf.keras.utils.register_keras_serializable(package="NutriBone")
class ConvNeXtPreprocess(KL.Layer):
    def call(self, x): return tf.keras.applications.convnext.preprocess_input(x)
    def get_config(self): return super().get_config()


CUSTOM_OBJECTS = {
    "FocalLoss":                FocalLoss,
    "WarmupCosineDecay":        WarmupCosineDecay,
    "WarmupCosine":             WarmupCosineDecay,
    "EfficientNetV2Preprocess": EfficientNetV2Preprocess,
    "DenseNetPreprocess":       DenseNetPreprocess,
    "ConvNeXtPreprocess":       ConvNeXtPreprocess,
}


from tensorflow.keras.applications import (DenseNet201 as _DN,
    EfficientNetV2B2 as _EFF, ConvNeXtTiny as _CNX)

def _load_head(x):
    x = KL.GlobalAveragePooling2D()(x); x = KL.BatchNormalization()(x)
    x = KL.Dense(512, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = KL.Dropout(0.4)(x)
    x = KL.Dense(256, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = KL.Dropout(0.3)(x)
    return KL.Dense(NUM_CLASSES, activation='softmax')(x)

def _build_for_transplant(name):
    kw = dict(weights=None, include_top=False, input_shape=(*IMG_SIZE, 3))
    inp = KL.Input(shape=(*IMG_SIZE, 3), name='input_image')
    if name == 'EfficientNetV2B2':
        x = EfficientNetV2Preprocess(name='eff_preprocess')(inp)
        base = _EFF(**kw, include_preprocessing=False)
    elif name == 'DenseNet201':
        x = DenseNetPreprocess(name='dense_preprocess')(inp)
        base = _DN(**kw)
    elif name == 'ConvNeXtTiny':
        x = ConvNeXtPreprocess(name='convnext_preprocess')(inp)
        base = _CNX(**kw)
    else:
        raise ValueError(f'Unknown backbone: {name}')
    return keras.Model(inp, _load_head(base(x)), name=name)


def safe_load_model(path: str, backbone_name: str) -> keras.Model:
    import zipfile, tempfile
    try:
        m = keras.models.load_model(path, custom_objects=CUSTOM_OBJECTS)
        log.info('  load OK (normal)    %s', os.path.basename(path))
        return m
    except (TypeError, KeyError, ValueError) as exc:
        log.warning('  normal load failed (%s) -> transplant...', type(exc).__name__)
    model = _build_for_transplant(backbone_name)
    with tempfile.TemporaryDirectory() as tmp:
        with zipfile.ZipFile(path, 'r') as zf:
            names = zf.namelist()
            wf = next((n for n in names if n.endswith('.weights.h5')), None)
            if wf is None:
                raise FileNotFoundError(f'No .weights.h5 in {path}. Contents: {names}')
            zf.extract(wf, tmp)
            model.load_weights(os.path.join(tmp, wf), by_name=False, skip_mismatch=False)
    log.info('  load OK (transplant) %s', os.path.basename(path))
    return model

In [6]:
def strip_exif_and_load(file_stream) -> np.ndarray:
    """Load image, strip EXIF metadata (privacy), return RGB uint8 array."""
    pil   = Image.open(file_stream).convert("RGB")
    clean = Image.new("RGB", pil.size)
    clean.paste(pil)
    return np.array(clean)


def validate_image(img: np.ndarray):
    h, w = img.shape[:2]
    if h < MIN_DIM or w < MIN_DIM:
        return f"Image too small ({w}×{h}). Minimum {MIN_DIM}×{MIN_DIM}."
    if h > 8000 or w > 8000:
        return f"Image too large ({w}×{h}). Maximum 8000×8000."
    return None


def apply_clahe(bgr: np.ndarray, clip: float = 2.0,
                grid: tuple = (8, 8)) -> np.ndarray:
    lab     = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    cl      = cv2.createCLAHE(clipLimit=clip, tileGridSize=grid)
    return cv2.cvtColor(cv2.merge([cl.apply(l), a, b]), cv2.COLOR_LAB2BGR)


def preprocess_for_model(rgb: np.ndarray) -> np.ndarray:
    """RGB uint8 → CLAHE → resize → float32 [0, 255] RGB (backbone normalises internally)."""
    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    if bgr.ndim == 2:
        bgr = cv2.cvtColor(bgr, cv2.COLOR_GRAY2BGR)
    bgr = apply_clahe(bgr)
    bgr = cv2.resize(bgr, IMG_SIZE)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).astype(np.float32)


def get_clahe_b64(rgb: np.ndarray) -> str:
    processed = preprocess_for_model(rgb)
    pil = Image.fromarray(processed.astype(np.uint8))
    buf = io.BytesIO()
    pil.save(buf, "JPEG", quality=88)
    return base64.b64encode(buf.getvalue()).decode()


def ndarray_to_b64(rgb_array: np.ndarray, fmt: str = "JPEG") -> str:
    pil = Image.fromarray(rgb_array.astype(np.uint8))
    buf = io.BytesIO()
    pil.save(buf, fmt, quality=88)
    return base64.b64encode(buf.getvalue()).decode()

In [7]:
tta_aug = keras.Sequential([
    KL.RandomRotation(0.07),
    KL.RandomTranslation(0.04, 0.04),
    KL.RandomZoom(0.07),
    KL.RandomFlip("horizontal"),
], name="tta")

In [8]:
def generate_gradcam(model: Model, rgb: np.ndarray, backbone_name: str):

    if backbone_name != "EfficientNetV2B2":
        return None

    arr = tf.constant(
        np.expand_dims(preprocess_for_model(rgb), 0),
        dtype=tf.float32
    )

    backbone = model.get_layer("efficientnetv2-b2")

    feature_extractor = tf.keras.Model(
        inputs=backbone.input,
        outputs=[
            backbone.get_layer("top_conv").output,
            backbone.output
        ]
    )

    with tf.GradientTape() as tape:

        conv_features, backbone_features = feature_extractor(arr)

        x = backbone_features

        for i in range(3, 10):
            x = model.layers[i](x)

        preds = x

        class_idx = tf.argmax(preds[0])
        loss = preds[:, class_idx]

    grads = tape.gradient(loss, conv_features)

    if grads is None:
        print("GradCAM gradients are None")
        return None

    weights = tf.reduce_mean(grads, axis=(1, 2))

    cam = tf.reduce_sum(
        conv_features * weights[:, tf.newaxis, tf.newaxis, :],
        axis=-1
    )

    cam = tf.nn.relu(cam)

    heatmap = cam[0].numpy()

    if np.max(heatmap) <= 0:
        print("Empty heatmap")
        return None

    heatmap /= np.max(heatmap)

    heatmap = cv2.resize(
        heatmap,
        IMG_SIZE,
        interpolation=cv2.INTER_CUBIC
    )

    heatmap_color = cv2.applyColorMap(
        np.uint8(255 * heatmap),
        cv2.COLORMAP_INFERNO
    )

    original = cv2.resize(
        cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR),
        IMG_SIZE
    )

    overlay = cv2.addWeighted(
        original,
        0.55,
        heatmap_color,
        0.45,
        0
    )

    return ndarray_to_b64(
        cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB),
        "JPEG"
    )

In [9]:
class EnsemblePredictor:
    def __init__(self):
        self.mode         = "demo"
        self.models: dict = {}
        self.temperatures: dict = {}
        self.ens_weights:  dict = {}
        self.model_info:   dict = {}

        if os.path.exists(CONFIG_PATH):
            log.info("Loading ensemble from %s", CONFIG_PATH)
            self._load_ensemble()
        else:
            log.info("No config — scanning for single model")
            self._load_single()

        log.info("Predictor ready (mode=%s)", self.mode)

    def _load_ensemble(self):
        with open(CONFIG_PATH) as f:
            cfg = json.load(f)

        self.temperatures = cfg.get("temperatures", {})
        self.ens_weights  = cfg.get("ensemble_weights", {})
        total = sum(self.ens_weights.values()) or 1.0
        self.ens_weights  = {k: v / total for k, v in self.ens_weights.items()}

        oof_auc = cfg.get("oof_auc", {})
        for bname, paths in cfg.get("fold_models", {}).items():
            if paths and os.path.exists(paths[0]):
                log.info("  Loading %s …", bname)
                m = safe_load_model(paths[0], bname)
                self.models[bname] = m
                self.model_info[bname] = {
                    "path":    os.path.basename(paths[0]),
                    "oof_auc": round(oof_auc.get(bname, 0), 4),
                    "weight":  round(self.ens_weights.get(bname, 0), 4),
                }

        if self.models:
            self.mode = "ensemble"
        else:
            self._load_single()

    def _load_single(self):
        for candidate in ["nutribone_final.keras", "nutribone_best.keras"]:
            if os.path.exists(candidate):
                m = safe_load_model(candidate, "model")
                self.models      = {"model": m}
                self.temperatures = {"model": 1.0}
                self.ens_weights  = {"model": 1.0}
                self.model_info   = {"model": {"path": candidate, "oof_auc": 0}}
                self.mode = "single"
                log.info("Single model loaded: %s", candidate)
                return
        log.warning("No model files found — demo mode")

    def _tta_predict(self, model, arr: np.ndarray, T: float) -> np.ndarray:
        batch = tf.constant(arr)
        acc   = np.zeros((1, NUM_CLASSES), dtype=np.float32)
        for _ in range(TTA_STEPS):
            aug  = tta_aug(batch, training=True).numpy()
            acc += model.predict(aug, verbose=0, batch_size=1)
        raw    = acc / TTA_STEPS
        logits = np.log(np.clip(raw, 1e-8, 1.0))
        return softmax(logits / max(T, 0.1), axis=1)

    def predict(self, rgb: np.ndarray) -> dict:
        timing = {}

        if self.mode == "demo":
            p   = np.random.dirichlet([4, 2, 1])
            idx = int(np.argmax(p))
            return {
                "label":      CATEGORIES[idx],
                "confidence": float(p[idx]),
                "scores":     {c: float(v) for c, v in zip(CATEGORIES, p)},
                "mode":       "demo",
                "timing":     {},
            }

        t0  = time.perf_counter()
        arr = np.expand_dims(preprocess_for_model(rgb), 0)
        timing["preprocessing_ms"] = round((time.perf_counter() - t0) * 1000, 1)

        ens       = np.zeros((1, NUM_CLASSES), dtype=np.float32)
        per_model = {}
        for bname, model in self.models.items():
            t1     = time.perf_counter()
            T      = self.temperatures.get(bname, 1.0)
            w      = self.ens_weights.get(bname, 1.0 / len(self.models))
            scaled = self._tta_predict(model, arr, T)
            ens   += w * scaled
            timing[f"{bname}_ms"] = round((time.perf_counter() - t1) * 1000, 1)
            per_model[bname] = {
                "scores":      {c: round(float(v), 4) for c, v in zip(CATEGORIES, scaled[0])},
                "weight":      round(w, 4),
                "temperature": round(T, 4),
            }

        ens /= ens.sum()
        idx  = int(np.argmax(ens[0]))

        return {
            "label":      CATEGORIES[idx],
            "confidence": round(float(ens[0, idx]), 4),
            "scores":     {c: round(float(v), 4) for c, v in zip(CATEGORIES, ens[0])},
            "per_model":  per_model,
            "mode":       self.mode,
            "timing":     timing,
        }

In [10]:
predictor = EnsemblePredictor()
app       = Flask(__name__)
CORS(app)

20:21:08 [INFO] Loading ensemble from nutribone_config.json
20:21:08 [INFO]   Loading EfficientNetV2B2 …
20:21:10 [INFO]   load OK (normal)    EfficientNetV2B2_fold1_best.keras
20:21:10 [INFO]   Loading DenseNet201 …
20:21:14 [INFO]   load OK (normal)    DenseNet201_fold1_best.keras
20:21:14 [INFO]   Loading ConvNeXtTiny …
20:21:16 [INFO]   load OK (normal)    ConvNeXtTiny_fold1_best.keras
20:21:16 [INFO] Predictor ready (mode=ensemble)


In [11]:
import cv2

img = cv2.imread("/home/adarsha007/Downloads/d4b53bd3e6c3c3b332eeb2050da868_big_gallery.jpg")
rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

In [12]:
first_name = list(predictor.models.keys())[0]
first_model = predictor.models[first_name]

print(first_name)

EfficientNetV2B2


In [13]:
grad = generate_gradcam(
    first_model,
    rgb,
    "EfficientNetV2B2"
)

print(type(grad))

<class 'str'>


In [14]:
@app.route("/")
def index():
    return render_template("index.html")


@app.route("/health")
def health():
    return jsonify({
        "status":     "ok",
        "mode":       predictor.mode,
        "models":     list(predictor.models.keys()),
        "model_info": predictor.model_info,
        "img_size":   list(IMG_SIZE),
        "tta_steps":  TTA_STEPS,
    })


@app.route("/clahe", methods=["POST"])
def clahe_preview():
    if "image" not in request.files:
        return jsonify({"error": "NO_IMAGE", "message": "No image provided"}), 400
    try:
        rgb = strip_exif_and_load(request.files["image"].stream)
        err = validate_image(rgb)
        if err:
            return jsonify({"error": "INVALID_IMAGE", "message": err}), 422
        h, w = rgb.shape[:2]
        return jsonify({"clahe_b64": get_clahe_b64(rgb), "original_size": [w, h]})
    except Exception:
        log.error(traceback.format_exc())
        return jsonify({"error": "PROCESSING_ERROR",
                        "message": "Could not process image"}), 500


@app.route("/predict", methods=["POST"])
def predict():
    if "image" not in request.files:
        return jsonify({"error": "NO_IMAGE", "message": "No image provided"}), 400

    f = request.files["image"]
    f.stream.seek(0, 2)
    size_mb = f.stream.tell() / (1024 * 1024)
    f.stream.seek(0)

    if size_mb > MAX_FILE_MB:
        return jsonify({
            "error": "FILE_TOO_LARGE",
            "message": f"Max {MAX_FILE_MB} MB allowed"
        }), 413

    t_start = time.perf_counter()

    try:
        rgb = strip_exif_and_load(f.stream)

        err = validate_image(rgb)
        if err:
            return jsonify({
                "error": "INVALID_IMAGE",
                "message": err
            }), 422

        result = predictor.predict(rgb)

        result["total_ms"] = round(
            (time.perf_counter() - t_start) * 1000, 1
        )
        result["gradcam"] = None
        result["clahe"] = get_clahe_b64(rgb)

        if predictor.mode in ("ensemble", "single"):
            first_name = list(predictor.models.keys())[0]
            first_model = predictor.models[first_name]

            backbone = first_model.get_layer("efficientnetv2-b2")

            print("\nBACKBONE LAYERS")
            print("=" * 80)

            for layer in backbone.layers[-20:]:
                print(layer.name, type(layer).__name__)

            result["gradcam"] = generate_gradcam(
                first_model,
                rgb,
                first_name
)

            result["gradcam_layer"] = "top_conv"

        prediction_history.appendleft({
            "id": str(uuid.uuid4())[:8],
            "timestamp": datetime.now().strftime("%H:%M:%S"),
            "label": result["label"],
            "confidence": result["confidence"],
            "scores": result["scores"],
            "thumbnail": result["clahe"],
        })

        return jsonify(result)

    except Exception:
        log.error(traceback.format_exc())
        return jsonify({
            "error": "PREDICTION_ERROR",
            "message": "Prediction failed — check server log"
        }), 500


@app.route("/history")
def history():
    return jsonify(list(prediction_history))

In [15]:
[x for x in globals() if "img" in x.lower() or "rgb" in x.lower()]

['IMG_SIZE', 'img', 'rgb']

In [16]:
for i, layer in enumerate(first_model.layers):
    print(i, layer.name, type(layer).__name__)

0 input_image InputLayer
1 eff_preprocess EfficientNetV2Preprocess
2 efficientnetv2-b2 Functional
3 global_average_pooling2d GlobalAveragePooling2D
4 batch_normalization BatchNormalization
5 dense Dense
6 dropout Dropout
7 dense_1 Dense
8 dropout_1 Dropout
9 dense_2 Dense


In [17]:
for i in range(3, 10):
    print(i, first_model.layers[i].name)

3 global_average_pooling2d
4 batch_normalization
5 dense
6 dropout
7 dense_1
8 dropout_1
9 dense_2


In [18]:
import tensorflow as tf

backbone = first_model.get_layer("efficientnetv2-b2")

feature_extractor = tf.keras.Model(
    inputs=backbone.input,
    outputs=[
        backbone.get_layer("top_conv").output,
        backbone.output
    ]
)

dummy = tf.random.normal((1, 384, 384, 3))

conv_features, backbone_features = feature_extractor(dummy)

print("conv_features:", conv_features.shape)
print("backbone_features:", backbone_features.shape)

conv_features: (1, 12, 12, 1408)
backbone_features: (1, 12, 12, 1408)


In [19]:
x = backbone_features

for i in range(3, 10):
    x = first_model.layers[i](x)

print("final prediction shape:", x.shape)

final prediction shape: (1, 3)


In [20]:
print("\nOUTER INPUTS")
print(first_model.inputs)

print("\nOUTER OUTPUTS")
print(first_model.outputs)


OUTER INPUTS
[<KerasTensor shape=(None, 384, 384, 3), dtype=float32, sparse=False, ragged=False, name=input_image>]

OUTER OUTPUTS
[<KerasTensor shape=(None, 3), dtype=float32, sparse=False, ragged=False, name=keras_tensor_751>]


In [21]:
if __name__ == "__main__":
    print()
    print("  NutriBone v2  ·  localhost:5000     ")
    print()
    app.run(host="0.0.0.0", port=5000, debug=False, threaded=False)


  NutriBone v2  ·  localhost:5000     

 * Serving Flask app '__main__'
 * Debug mode: off


20:21:18 [INFO] WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.0.243:5000
20:21:18 [INFO] Press CTRL+C to quit
20:21:26 [INFO] 127.0.0.1 - - [15/Jul/2026 20:21:26] "GET / HTTP/1.1" 200 -
20:21:26 [INFO] 127.0.0.1 - - [15/Jul/2026 20:21:26] "GET /health HTTP/1.1" 200 -
20:21:26 [INFO] 127.0.0.1 - - [15/Jul/2026 20:21:26] "GET /history HTTP/1.1" 200 -
20:21:35 [INFO] 127.0.0.1 - - [15/Jul/2026 20:21:35] "POST /clahe HTTP/1.1" 200 -



BACKBONE LAYERS
block6i_drop Dropout
block6i_add Add
block6j_expand_conv Conv2D
block6j_expand_bn BatchNormalization
block6j_expand_activation Activation
block6j_dwconv2 DepthwiseConv2D
block6j_bn BatchNormalization
block6j_activation Activation
block6j_se_squeeze GlobalAveragePooling2D
block6j_se_reshape Reshape
block6j_se_reduce Conv2D
block6j_se_expand Conv2D
block6j_se_excite Multiply
block6j_project_conv Conv2D
block6j_project_bn BatchNormalization
block6j_drop Dropout
block6j_add Add
top_conv Conv2D
top_bn BatchNormalization
top_activation Activation


20:22:21 [INFO] 127.0.0.1 - - [15/Jul/2026 20:22:21] "POST /predict HTTP/1.1" 200 -
20:22:21 [INFO] 127.0.0.1 - - [15/Jul/2026 20:22:21] "GET /history HTTP/1.1" 200 -
